# Concurrency, Parallelism & Asyncio: Beginner Guide

### 📌 Overview
Master **Concurrency, Parallelism & Asyncio: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Multi-Threading**: Covers `threading.Thread` and `threading.Lock`.
- **Execution Pools**: Covers `concurrent.futures.ThreadPoolExecutor` and `ProcessPoolExecutor`.
- **Asynchronous Event Loop**: Covers `async def`, `await`, `asyncio.run()`, and `asyncio.gather()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import functools
import contextlib
import asyncio
import threading
from dataclasses import dataclass
from typing import List, Dict, Optional, Union, Protocol, Literal, Final, TypedDict, Callable, TypeVar

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Multi-Threading: `threading.Thread`
- **What it does:** Spawns OS threads sharing Python heap memory space (suitable for I/O-bound tasks).
- **Syntax:** `t = threading.Thread(target=func, args=(...)); t.start(); t.join()`
- **Operation:** `def worker_audit(tx_id):`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [2]:
def worker_audit(tx_id):
    print(f'Thread auditing {tx_id}')

t1 = threading.Thread(target=worker_audit, args=(transactions[0]['transaction_id'],))
t1.start()
t1.join()
print('Thread worker finished.')

Thread auditing TX109326
Thread worker finished.


### 🔹 Thread Synchronization: `threading.Lock`
- **What it does:** Mutual exclusion lock preventing race conditions on shared memory across threads.
- **Syntax:** `with lock: shared_state += 1`
- **Operation:** `total_balance = 0.0`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [3]:
total_balance = 0.0
lock = threading.Lock()

def thread_safe_deposit(amt):
    global total_balance
    with lock:
        total_balance += amt

threads = [threading.Thread(target=thread_safe_deposit, args=(float(transactions[i]['transaction_amount']),)) for i in range(3)]
for t in threads: t.start()
for t in threads: t.join()
print(f'Thread-Safe Consolidated Balance: ${total_balance:,.2f}')

Thread-Safe Consolidated Balance: $2,490.97


### 🔹 Thread Pool Executor: `ThreadPoolExecutor`
- **What it does:** High-level thread pool manager mapping worker functions across threads.
- **Syntax:** `with ThreadPoolExecutor() as ex: results = ex.map(fn, items)`
- **Operation:** `from concurrent.futures import ThreadPoolExecutor`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [4]:
from concurrent.futures import ThreadPoolExecutor
def process_item(t):
    return f"{t['transaction_id']}: Verified {t['card_type']}"

with ThreadPoolExecutor(max_workers=3) as pool:
    results = list(pool.map(process_item, transactions[:3]))
print('ThreadPoolExecutor results:', results)

ThreadPoolExecutor results: ['TX109326: Verified Visa', 'TX106376: Verified Visa', 'TX103301: Verified Visa']


### 🔹 Process Pool Executor: `ProcessPoolExecutor`
- **What it does:** Spawns independent OS Python processes bypassing the GIL for CPU-bound computation.
- **Syntax:** `with ProcessPoolExecutor() as ex: ...`
- **Operation:** `from concurrent.futures import ProcessPoolExecutor`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [5]:
from concurrent.futures import ProcessPoolExecutor
print('ProcessPoolExecutor: Bypasses CPython GIL by spawning separate OS processes.')

ProcessPoolExecutor: Bypasses CPython GIL by spawning separate OS processes.


### 🔹 Asynchronous Coroutines: `async def` & `await`
- **What it does:** Defines non-blocking coroutines yielding control back to event loop on async I/O.
- **Syntax:** `async def fetch(): await asyncio.sleep(0.01)`
- **Operation:** `async def simulate_async_api(tx_id):`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [6]:
async def simulate_async_api(tx_id):
    await asyncio.sleep(0.01) # Non-blocking async sleep
    return f'{tx_id}: APPROVED'

print('Coroutine function defined.')

Coroutine function defined.


### 🔹 Event Loop Execution: `asyncio.run()`
- **What it does:** Creates a new event loop, executes the main coroutine, and closes the loop.
- **Syntax:** `asyncio.run(main())`
- **Operation:** `import nest_asyncio`
- **Key Note:** Always remember to include `self` as the first parameter in instance methods so Python knows which object instance is executing.

In [7]:
import nest_asyncio
nest_asyncio.apply()
async def main():
    return await simulate_async_api(transactions[0]['transaction_id'])
print('asyncio.run result:', asyncio.run(main()))


asyncio.run result: TX109326: APPROVED


### 🔹 Concurrent Task Gathering: `asyncio.gather()`
- **What it does:** Runs multiple async coroutines concurrently on single-threaded event loop.
- **Syntax:** `await asyncio.gather(*tasks)`
- **Operation:** `import nest_asyncio`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [8]:
import nest_asyncio
nest_asyncio.apply()
async def run_batch():
    tasks = [simulate_async_api(t['transaction_id']) for t in transactions[:3]]
    return await asyncio.gather(*tasks)
print('asyncio.gather results:', asyncio.run(run_batch()))


asyncio.gather results: ['TX109326: APPROVED', 'TX106376: APPROVED', 'TX103301: APPROVED']


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: When to choose Threading vs Multiprocessing vs Asyncio
- **Objective:** Q1: When to choose Threading vs Multiprocessing vs Asyncio
- **Approach:** Explain trade-offs: (1) Threading for I/O blocking calls; (2) Multiprocessing for CPU bound GIL bypass; (3) Asyncio for high concurrency non-blocking network sockets.
- **Syntax:** `ProcessPoolExecutor` vs `ThreadPoolExecutor` vs `asyncio`

In [9]:
print('CPU-Bound: Multiprocessing (separate OS processes).')
print('I/O-Bound High Concurrency: Asyncio (event loop coroutines).')
print('I/O-Bound Blocking APIs: Threading.')

CPU-Bound: Multiprocessing (separate OS processes).
I/O-Bound High Concurrency: Asyncio (event loop coroutines).
I/O-Bound Blocking APIs: Threading.
